In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
import os

In [3]:
TYPE = ["cvrp", 
        "ovrp", "ovrpb", "ovrpbl", "ovrpbltw", "ovrpbtw", "ovrpl", "ovrpltw", "ovrptw",
        "vrpb", "vrpbl", "vrpbltw", "vrpbtw", "vrpl", "vrpltw", "vrptw"]
NAME = ["ortools", "pyvrp", "mtpomo", "mvmoe", "rf-moe", "rf-pomo", "rf-transformer"]
BASELINE_NAMES = ["ortools", "pyvrp"]
COMPARISON_NAMES = ["mtpomo", "mvmoe", "rf-moe", "rf-pomo", "rf-transformer"]
NUM_NODES = [50, 100]

In [4]:
def load_costs(file_path):
    """加载solution文件并返回costs"""
    try:
        data = np.load(file_path)
        costs = data['costs']  # shape: (1000,)
        return costs
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None
def load_time(file_path):
    """加载solution文件并返回costs"""
    try:
        data = np.load(file_path)
        costs = data['time']  # shape: (1000,)
        return costs
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None

In [ ]:
DATA_ROOT = "data"
for num in NUM_NODES:
    print(f"\n{'='*60}")
    print(f"Processing {num}-node instances")
    print(f"{'='*60}")
    
    all_records = []
    
    # 遍历所有类型和模型
    for vrp_type in TYPE:
        for name in BASELINE_NAMES:
            # 构建文件路径
            file_path = Path(DATA_ROOT) / vrp_type / "test" / f"{num}_sol_{name}.npz"
            
            print(f"  Loading: {file_path}")
            
            if not file_path.exists():
                print(f"File not found")
                continue
            
            # 加载costs
            costs = load_costs(file_path)
            
            if costs is None:
                continue
            
            # 为每个实例创建记录
            for instance_id in range(len(costs)):
                record = {
                    'problem_id': f"{vrp_type}_{num}_{instance_id:04d}",
                    'type': vrp_type,
                    'model': name,
                    'cost': costs[instance_id]
                }
                all_records.append(record)
            
            print(f"    ✓ Added {len(costs)} records")
    
    # 创建DataFrame
    df = pd.DataFrame(all_records)
    
    # 检查数据完整性
    print(f"\n{'='*60}")
    print(f"Summary for {num}-node instances:")
    print(f"  Total records: {len(df)}")
    expected = len(TYPE) * len(BASELINE_NAMES) * 1000
    print(f"  Expected: {expected}")
    print(f"  Completion: {len(df)/expected*100:.1f}%")
    
    # 检查每个组合的实例数
    counts = df.groupby(['type', 'model']).size().reset_index(name='count')
    incomplete = counts[counts['count'] != 1000]
    if len(incomplete) > 0:
        print(f"\n  ⚠️ Incomplete combinations:")
        print(incomplete.to_string(index=False))
    else:
        print(f"\n  ✓ All combinations have 1000 instances")
    
    # 查看前几行
    print(f"\n  First 10 rows:")
    print(df.head(10))
    
    # 保存CSV
    output_file = f"vrp_results_{num}nodes.csv"
    df.to_csv(output_file, index=False)
    print(f"\n  ✓ Saved to {output_file}")
    
    # 显示文件大小
    file_size = os.path.getsize(output_file) / (1024 * 1024)
    print(f"  File size: {file_size:.2f} MB")
    
    # 可选：显示一些统计信息
    print(f"\n  Cost statistics by model:")
    cost_stats = df.groupby('model')['cost'].agg(['count', 'mean', 'std', 'min', 'max'])
    print(cost_stats.round(4))

print(f"\n{'='*60}")
print("All done!")


Processing 50-node instances
  Loading: data/cvrp/test/50_sol_ortools.npz
    ✓ Added 1000 records
  Loading: data/cvrp/test/50_sol_pyvrp.npz
    ✓ Added 1000 records
  Loading: data/ovrp/test/50_sol_ortools.npz
    ✓ Added 1000 records
  Loading: data/ovrp/test/50_sol_pyvrp.npz
    ✓ Added 1000 records
  Loading: data/ovrpb/test/50_sol_ortools.npz
    ✓ Added 1000 records
  Loading: data/ovrpb/test/50_sol_pyvrp.npz
    ✓ Added 1000 records
  Loading: data/ovrpbl/test/50_sol_ortools.npz
    ✓ Added 1000 records
  Loading: data/ovrpbl/test/50_sol_pyvrp.npz
    ✓ Added 1000 records
  Loading: data/ovrpbltw/test/50_sol_ortools.npz
    ✓ Added 1000 records
  Loading: data/ovrpbltw/test/50_sol_pyvrp.npz
    ✓ Added 1000 records
  Loading: data/ovrpbtw/test/50_sol_ortools.npz
    ✓ Added 1000 records
  Loading: data/ovrpbtw/test/50_sol_pyvrp.npz
    ✓ Added 1000 records
  Loading: data/ovrpl/test/50_sol_ortools.npz
    ✓ Added 1000 records
  Loading: data/ovrpl/test/50_sol_pyvrp.npz
    ✓ A

In [13]:
DATA_ROOT = "Results"
for num in NUM_NODES:
    print(f"\n{'='*60}")
    print(f"Processing {num}-node comparison models")
    print(f"{'='*60}")
    
    all_records = []
    
    # 遍历所有类型和比较模型
    for vrp_type in TYPE:
        for name in COMPARISON_NAMES:
            # 构建文件路径: Results/results_[NUM]/solutions/[NAME]/[TYPE]_[NUM]_sol.npz
            file_path = Path(DATA_ROOT) / f"results_{num}" / "solutions" / name / f"{vrp_type}_{num}_sol.npz"
            
            print(f"  Loading: {file_path}")
            
            if not file_path.exists():
                print(f"File not found")
                continue
            
            # 加载costs
            costs = load_costs(file_path)
            
            if costs is None:
                continue
            
            # 确保有1000个实例
            if len(costs) != 1000:
                print(f"    ⚠️ Expected 1000 instances, got {len(costs)}")
            
            # 为每个实例创建记录
            for instance_id in range(len(costs)):
                record = {
                    'problem_id': f"{vrp_type}_{num}_{instance_id:04d}",
                    'type': vrp_type,
                    'model': name,
                    'cost': costs[instance_id]
                }
                all_records.append(record)
            
            print(f"    ✓ Added {len(costs)} records for {vrp_type}_{num}_{name}")
    
    # 创建新数据的DataFrame
    new_df = pd.DataFrame(all_records)
    
    # 读取已有的baseline文件（如果存在）
    existing_file = f"vrp_results_{num}nodes.csv"
    if os.path.exists(existing_file):
        print(f"\n  Reading existing file: {existing_file}")
        existing_df = pd.read_csv(existing_file)
        print(f"  Existing records: {len(existing_df)}")
        
        # 合并数据
        combined_df = pd.concat([existing_df, new_df], ignore_index=True)
        print(f"  Combined records: {len(combined_df)}")
    else:
        print(f"\n  ⚠️ Baseline file not found, creating new file")
        combined_df = new_df
    
    # 保存合并后的数据
    combined_df.to_csv(existing_file, index=False)
    print(f"\n  ✓ Saved to {existing_file}")
    
    # 显示文件大小
    file_size = os.path.getsize(existing_file) / (1024 * 1024)
    print(f"  File size: {file_size:.2f} MB")
    
    # 显示统计信息
    print(f"\n  Current data summary for {num}-node instances:")
    print(f"  Total records: {len(combined_df)}")
    print(f"  Models in file: {combined_df['model'].unique()}")
    
    # 按模型统计
    model_stats = combined_df.groupby('model')['cost'].agg(['count', 'mean', 'std', 'min', 'max'])
    print(f"\n  Cost statistics by model:")
    print(model_stats.round(4))

print(f"\n{'='*60}")
print("All done!")



Processing 50-node comparison models
  Loading: Results/results_50/solutions/mtpomo/cvrp_50_sol.npz
    ✓ Added 1000 records for cvrp_50_mtpomo
  Loading: Results/results_50/solutions/mvmoe/cvrp_50_sol.npz
    ✓ Added 1000 records for cvrp_50_mvmoe
  Loading: Results/results_50/solutions/rf-moe/cvrp_50_sol.npz
    ✓ Added 1000 records for cvrp_50_rf-moe
  Loading: Results/results_50/solutions/rf-pomo/cvrp_50_sol.npz
    ✓ Added 1000 records for cvrp_50_rf-pomo
  Loading: Results/results_50/solutions/rf-transformer/cvrp_50_sol.npz
    ✓ Added 1000 records for cvrp_50_rf-transformer
  Loading: Results/results_50/solutions/mtpomo/ovrp_50_sol.npz
    ✓ Added 1000 records for ovrp_50_mtpomo
  Loading: Results/results_50/solutions/mvmoe/ovrp_50_sol.npz
    ✓ Added 1000 records for ovrp_50_mvmoe
  Loading: Results/results_50/solutions/rf-moe/ovrp_50_sol.npz
    ✓ Added 1000 records for ovrp_50_rf-moe
  Loading: Results/results_50/solutions/rf-pomo/ovrp_50_sol.npz
    ✓ Added 1000 records for

In [14]:
for num in NUM_NODES:
    file_path = f"vrp_results_{num}nodes.csv"
    
    if not os.path.exists(file_path):
        print(f"⚠️ File not found: {file_path}")
        continue
    
    print(f"\n{'='*60}")
    print(f"Processing {file_path}")
    print(f"{'='*60}")
    
    # 读取数据
    df = pd.read_csv(file_path)
    print(f"Original records: {len(df)}")
    print(f"Original cost range: [{df['cost'].min():.4f}, {df['cost'].max():.4f}]")
    print(f"Original cost statistics:")
    print(f"  Mean: {df['cost'].mean():.4f}")
    print(f"  Std: {df['cost'].std():.4f}")
    
    # 取绝对值并保留五位小数
    df['cost'] = df['cost'].abs().round(5)
    
    print(f"\nAfter transformation:")
    print(f"New cost range: [{df['cost'].min():.5f}, {df['cost'].max():.5f}]")
    print(f"New cost statistics:")
    print(f"  Mean: {df['cost'].mean():.5f}")
    print(f"  Std: {df['cost'].std():.5f}")
    
    # 检查不同模型的变化
    print(f"\nCost changes by model:")
    for model in df['model'].unique():
        model_df = df[df['model'] == model]
        print(f"  {model}: mean={model_df['cost'].mean():.5f}, std={model_df['cost'].std():.5f}")
    
    # 保存处理后的数据（覆盖原文件）
    df.to_csv(file_path, index=False)
    print(f"\n✓ Saved to {file_path}")
    
    # 显示文件大小
    file_size = os.path.getsize(file_path) / (1024 * 1024)
    print(f"File size: {file_size:.2f} MB")

print(f"\n{'='*60}")
print("All done! All costs now absolute values with 5 decimal places.")


Processing vrp_results_50nodes.csv
Original records: 112000
Original cost range: [-29.1211, 29.7094]
Original cost statistics:
  Mean: 5.0186
  Std: 11.1952

After transformation:
New cost range: [5.03700, 29.70940]
New cost statistics:
  Mean: 11.51319
  Std: 4.23852

Cost changes by model:
  ortools: mean=11.40187, std=4.19691
  pyvrp: mean=11.32924, std=4.17664
  mtpomo: mean=11.59279, std=4.26204
  mvmoe: mean=11.58220, std=4.26581
  rf-moe: mean=11.56733, std=4.25807
  rf-pomo: mean=11.56240, std=4.24886
  rf-transformer: mean=11.55647, std=4.25363

✓ Saved to vrp_results_50nodes.csv
File size: 4.00 MB

Processing vrp_results_100nodes.csv
Original records: 112000
Original cost range: [-45.9591, 46.7055]
Original cost statistics:
  Mean: 7.9638
  Std: 17.7988

After transformation:
New cost range: [8.03200, 46.70545]
New cost statistics:
  Mean: 18.18824
  Std: 7.02902

Cost changes by model:
  ortools: mean=18.07977, std=6.95085
  pyvrp: mean=17.70562, std=6.86486
  mtpomo: mean=

In [ ]:
DATA_ROOT = "instance_time"
for num in NUM_NODES:
    print(f"\n{'='*60}")
    print(f"Adding runtime data for {num}-node instances")
    print(f"{'='*60}")
    
    # 读取现有的CSV文件
    csv_file = f"vrp_results_{num}nodes.csv"
    
    if not os.path.exists(csv_file):
        print(f"⚠️ File not found: {csv_file}")
        continue
    
    df = pd.read_csv(csv_file)
    print(f"Original records: {len(df)}")
    print(f"Current columns: {df.columns.tolist()}")
    
    # 初始化runtime列（如果不存在）
    if 'runtime' not in df.columns:
        df['runtime'] = None
    
    # 记录添加的runtime数量
    runtime_added = 0
    
    # 遍历所有baseline模型和类型
    for vrp_type in TYPE:
        for name in BASELINE_NAMES:
            # 构建文件路径: instance_time/[NUM]/[NAME]/[TYPE].npz
            file_path = Path(DATA_ROOT) / str(num) / name / f"{vrp_type}.npz"
            
            print(f"  Loading runtime: {file_path}")
            
            if not file_path.exists():
                print(f" File not found")
                continue
            
            # 加载runtimes
            runtimes = load_time(file_path)
            
            if runtimes is None:
                continue
            
            # 确保有1000个实例
            if len(runtimes) != 1000:
                print(f"Expected 1000 instances, got {len(runtimes)}")
            
            # 更新DataFrame中对应的runtime列
            for instance_id in range(len(runtimes)):
                problem_id = f"{vrp_type}_{num}_{instance_id:04d}"
                mask = (df['problem_id'] == problem_id) & (df['model'] == name)
                
                if mask.any():
                    df.loc[mask, 'runtime'] = runtimes[instance_id]
                    runtime_added += 1
                else:
                    print(f"No matching record found for {problem_id} with model {name}")
            
            print(f"  Added {len(runtimes)} runtime records for {vrp_type}_{num}_{name}")
    
    # 检查runtime添加情况
    print(f"\n{'='*60}")
    print(f"Runtime addition summary:")
    print(f"  Runtime records added: {runtime_added}")
    
    # 检查baseline模型是否都有runtime
    baseline_records = df[df['model'].isin(BASELINE_NAMES)]
    expected_runtime_count = len(baseline_records)
    actual_runtime_count = baseline_records['runtime'].notna().sum()
    
    print(f"  Expected runtime records (baseline models): {expected_runtime_count}")
    print(f"  Actual runtime records added: {actual_runtime_count}")
    
    if actual_runtime_count < expected_runtime_count:
        print(f"  ⚠️ Missing {expected_runtime_count - actual_runtime_count} runtime values")
        # 显示缺失的模型和类型
        missing = baseline_records[baseline_records['runtime'].isna()]
        if len(missing) > 0:
            print(f"  Missing examples:")
            print(missing[['problem_id', 'type', 'model']].head(10))
    
    # 统计runtime信息
    if actual_runtime_count > 0:
        print(f"\n  Runtime statistics:")
        runtime_stats = df[df['runtime'].notna()].groupby('model')['runtime'].agg(['count', 'mean', 'std', 'min', 'max'])
        print(runtime_stats.round(5))
    
    # 保存更新后的CSV
    df.to_csv(csv_file, index=False)
    print(f"\n✓ Saved to {csv_file}")
    
    # 显示文件大小
    file_size = os.path.getsize(csv_file) / (1024 * 1024)
    print(f"File size: {file_size:.2f} MB")
    
    # 显示前几行查看效果
    print(f"\n  Sample rows with runtime:")
    sample_df = df[df['model'].isin(BASELINE_NAMES)].head(5)
    print(sample_df[['problem_id', 'model', 'cost', 'runtime']])

print(f"\n{'='*60}")
print("All done! Runtime data added for baseline models.")


Adding runtime data for 50-node instances
Original records: 112000
Current columns: ['problem_id', 'type', 'model', 'cost']
  Loading runtime: instance_time/50/ortools/cvrp.npz
  Added 1000 runtime records for cvrp_50_ortools
  Loading runtime: instance_time/50/pyvrp/cvrp.npz
  Added 1000 runtime records for cvrp_50_pyvrp
  Loading runtime: instance_time/50/ortools/ovrp.npz
  Added 1000 runtime records for ovrp_50_ortools
  Loading runtime: instance_time/50/pyvrp/ovrp.npz
  Added 1000 runtime records for ovrp_50_pyvrp
  Loading runtime: instance_time/50/ortools/ovrpb.npz
  Added 1000 runtime records for ovrpb_50_ortools
  Loading runtime: instance_time/50/pyvrp/ovrpb.npz
  Added 1000 runtime records for ovrpb_50_pyvrp
  Loading runtime: instance_time/50/ortools/ovrpbl.npz
  Added 1000 runtime records for ovrpbl_50_ortools
  Loading runtime: instance_time/50/pyvrp/ovrpbl.npz
  Added 1000 runtime records for ovrpbl_50_pyvrp
  Loading runtime: instance_time/50/ortools/ovrpbltw.npz
  Adde

In [7]:
DATA_ROOT = "instance_time1"
num = 100
print(f"\n{'='*60}")
print(f"Adding runtime data for {num}-node instances (all models)")
print(f"{'='*60}")

# 读取现有的CSV文件
csv_file = f"vrp_results_{num}nodes.csv"

if not os.path.exists(csv_file):
    print(f"⚠️ File not found: {csv_file}")
else:
    df = pd.read_csv(csv_file)
    print(f"Original records: {len(df)}")
    print(f"Models in file: {df['model'].unique().tolist()}")

    # 确保runtime列存在
    if 'runtime' not in df.columns:
        df['runtime'] = None

    # 记录添加的runtime数量
    runtime_added = 0
    missing_files = []

    # 遍历所有类型和所有模型
    for vrp_type in TYPE:
        for model_name in COMPARISON_NAMES:
            # 构建文件路径: instance_time/[NUM]/[NAME]/[TYPE].npz
            file_path = Path(DATA_ROOT) / str(num) / model_name / f"{vrp_type}.npz"
            
            # 检查文件是否存在
            if not file_path.exists():
                if model_name not in ["ortools", "pyvrp"]:  # 只记录comparison模型的缺失
                    missing_files.append(str(file_path))
                continue
            
            # 加载runtimes
            runtimes = load_time(file_path)
            
            if runtimes is None:
                continue
            
            # 确保有1000个实例
            if len(runtimes) != 1000:
                print(f" {model_name}/{vrp_type}: Expected 1000 instances, got {len(runtimes)}")
            
            # 更新DataFrame中对应的runtime列
            for instance_id in range(len(runtimes)):
                problem_id = f"{vrp_type}_{num}_{instance_id:04d}"
                mask = (df['problem_id'] == problem_id) & (df['model'] == model_name)
                
                if mask.any():
                    df.loc[mask, 'runtime'] = runtimes[instance_id]
                    runtime_added += 1
                else:
                    # 这种情况理论上不应该发生，因为problem_id应该都存在
                    if instance_id < 5:  # 只打印前几个警告
                        print(f" No matching record found for {problem_id} with model {model_name}")
            
            print(f"    ✓ Added {len(runtimes)} runtime records for {model_name}/{vrp_type}")

# 检查runtime添加情况
print(f"\n{'='*60}")
print(f"Runtime addition summary for {num}-node instances:")
print(f"  Runtime records added: {runtime_added}")

# 显示缺失runtime的文件
if missing_files:
    print(f"\n  ⚠️ Missing runtime files ({len(missing_files)}):")
    for file in missing_files[:10]:  # 只显示前10个
        print(f"    {file}")
    if len(missing_files) > 10:
        print(f"    ... and {len(missing_files) - 10} more")

# 统计runtime信息
if runtime_added > 0:
    print(f"\n  Runtime statistics by model:")
    runtime_stats = df[df['runtime'].notna()].groupby('model')['runtime'].agg(['count', 'mean', 'std', 'min', 'max'])
    print(runtime_stats.round(5))

# 保存更新后的CSV
df.to_csv(csv_file, index=False)
print(f"\n✓ Saved to {csv_file}")

# 显示文件大小
file_size = os.path.getsize(csv_file) / (1024 * 1024)
print(f"File size: {file_size:.2f} MB")

# 显示前几行查看效果
print(f"\n  Sample rows with runtime (all models):")
sample_df = df.head(10)
print(sample_df[['problem_id', 'model', 'cost', 'runtime']])

print(f"\n{'='*60}")
print("All done! Runtime data added for all models.")


Adding runtime data for 100-node instances (all models)
Original records: 112000
Models in file: ['ortools', 'pyvrp', 'mtpomo', 'mvmoe', 'rf-moe', 'rf-pomo', 'rf-transformer']
    ✓ Added 1000 runtime records for mtpomo/cvrp
    ✓ Added 1000 runtime records for mvmoe/cvrp
    ✓ Added 1000 runtime records for rf-moe/cvrp
    ✓ Added 1000 runtime records for rf-pomo/cvrp
    ✓ Added 1000 runtime records for rf-transformer/cvrp
    ✓ Added 1000 runtime records for mtpomo/ovrp
    ✓ Added 1000 runtime records for mvmoe/ovrp
    ✓ Added 1000 runtime records for rf-moe/ovrp
    ✓ Added 1000 runtime records for rf-pomo/ovrp
    ✓ Added 1000 runtime records for rf-transformer/ovrp
    ✓ Added 1000 runtime records for mtpomo/ovrpb
    ✓ Added 1000 runtime records for mvmoe/ovrpb
    ✓ Added 1000 runtime records for rf-moe/ovrpb
    ✓ Added 1000 runtime records for rf-pomo/ovrpb
    ✓ Added 1000 runtime records for rf-transformer/ovrpb
    ✓ Added 1000 runtime records for mtpomo/ovrpbl
    ✓ Ad

In [8]:
def reshape_data_for_nodes(num_nodes):
    """将指定节点数的数据重塑为宽格式"""
    
    # 读取原始数据
    input_file = f"vrp_results_{num_nodes}nodes.csv"
    output_file = f"vrp_results_{num_nodes}nodes_wide.csv"
    
    if not Path(input_file).exists():
        print(f"⚠️ File not found: {input_file}")
        return None
    
    print(f"\n{'='*60}")
    print(f"Processing {num_nodes}-node instances")
    print(f"{'='*60}")
    
    # 读取数据
    df = pd.read_csv(input_file)
    print(f"Original records: {len(df)}")
    print(f"Original columns: {df.columns.tolist()}")
    print(f"Unique instances: {df['problem_id'].nunique()}")
    print(f"Unique models: {df['model'].unique().tolist()}")
    
    # 创建 (cost, time) 对
    df['cost_time_pair'] = df.apply(lambda row: (row['cost'], row['runtime']), axis=1)
    
    # 透视数据：每个instance_id一行，每个model一列
    pivot_df = df.pivot_table(
        index=['problem_id', 'type'],
        columns='model',
        values='cost_time_pair',
        aggfunc='first'  # 每个组合应该只有一个值
    ).reset_index()
    
    # 重命名列
    pivot_df.columns.name = None  # 移除列名
    pivot_df = pivot_df.rename_axis(None, axis=1)
    
    # 确保所有模型都有列（如果某个模型没有数据，创建空列）
    for model in NAME:
        if model not in pivot_df.columns:
            pivot_df[model] = None
    
    # 重新排列列顺序
    column_order = ['problem_id', 'type'] + NAME
    pivot_df = pivot_df[column_order]
    
    # 统计信息
    print(f"\nWide format records: {len(pivot_df)}")
    print(f"Expected instances: {len(TYPE) * 1000}")
    print(f"Actual instances: {len(pivot_df)}")
    
    # 检查缺失情况
    print(f"\nMissing data per model:")
    for model in NAME:
        missing = pivot_df[model].isna().sum()
        total = len(pivot_df)
        print(f"  {model}: {missing}/{total} ({missing/total*100:.1f}%)")
    
    
    # 保存宽格式数据
    pivot_df.to_csv(output_file, index=False)
    print(f"\n✓ Saved to {output_file}")
    
    # 显示文件大小
    file_size = os.path.getsize(output_file) / (1024 * 1024)
    print(f"File size: {file_size:.2f} MB")
    
    # 可选：保存为parquet格式（更高效）
    parquet_file = f"vrp_results_{num_nodes}nodes_wide.parquet"
    pivot_df.to_parquet(parquet_file, index=False)
    print(f"✓ Also saved to {parquet_file}")
    
    return pivot_df

# 处理50和100节点的数据
for num in NUM_NODES:
    wide_df = reshape_data_for_nodes(num)
    
    # 额外保存一个json格式，方便查看某个实例的完整信息
    if wide_df is not None:
        sample_instance = wide_df.iloc[0]
        print(f"\nSample instance {sample_instance['problem_id']}:")
        for model in NAME:
            if pd.notna(sample_instance[model]):
                cost, runtime = sample_instance[model]
                print(f"  {model}: cost={cost:.4f}, runtime={runtime:.4f}")

print(f"\n{'='*60}")
print("All done! Data reshaped to wide format.")


Processing 50-node instances
Original records: 112000
Original columns: ['problem_id', 'type', 'model', 'cost', 'runtime']
Unique instances: 16000
Unique models: ['ortools', 'pyvrp', 'mtpomo', 'mvmoe', 'rf-moe', 'rf-pomo', 'rf-transformer']

Wide format records: 16000
Expected instances: 16000
Actual instances: 16000

Missing data per model:
  ortools: 0/16000 (0.0%)
  pyvrp: 0/16000 (0.0%)
  mtpomo: 0/16000 (0.0%)
  mvmoe: 0/16000 (0.0%)
  rf-moe: 0/16000 (0.0%)
  rf-pomo: 0/16000 (0.0%)
  rf-transformer: 0/16000 (0.0%)

✓ Saved to vrp_results_50nodes_wide.csv
File size: 3.48 MB
✓ Also saved to vrp_results_50nodes_wide.parquet

Sample instance cvrp_50_0000:
  ortools: cost=9.0818, runtime=10.0149
  pyvrp: cost=8.9000, runtime=10.0076
  mtpomo: cost=9.0307, runtime=0.3994
  mvmoe: cost=8.9890, runtime=0.4697
  rf-moe: cost=9.0430, runtime=0.4730
  rf-pomo: cost=8.9793, runtime=0.3670
  rf-transformer: cost=9.0285, runtime=0.3771

Processing 100-node instances
Original records: 112000
